# AI Hub 「농산물 품질(QC) 이미지」 — 사과 데이터 → 팀 공유 드라이브

이 노트북은 AI Hub에서 **사과(부사·양광) 파일만** 골라 받아 팀 Google Drive
`smart_apple_apc/data/raw/aihub_qc_apple/` 에 저장합니다. (PC에 받을 필요 없음)

## 실행 전 준비 (최초 1회, 본인이 직접)
1. [AI Hub](https://aihub.or.kr) 로그인 → [농산물 품질(QC) 이미지](https://aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&aihubDataSe=realm&dataSetSn=149) → **다운로드(데이터 신청)** → 승인 대기
2. AI Hub → AI 데이터 찾기 → **AI 허브 오픈 API** 에서 **API key 발급** → 회원 메일로 받은 키 확인
3. 이 노트북은 **드라이브 소유자(장준원) 계정**으로 실행 (공유 폴더가 `내 드라이브`에 있어야 경로가 맞음)

## 받는 파일
| 구분 | 내용 | 용량 |
|---|---|---|
| `labels` | 라벨(JSON) Training+Validation 12개 | 약 14MB |
| `val` | 검증 이미지 6개 (부사/양광 × L/M/S) | 약 3.7GB |
| `train` | 학습 이미지 6개 | 약 24.3GB |

> 💡 처음엔 `labels` + `val`만 받아서 파이프라인을 먼저 만들고, `train`은 나중에 받는 것을 추천합니다.
> 파일명의 L/M/S 가 특/상/보통에 대응하는지는 **라벨 JSON으로 확인** 후 `data/README.md`에 기록하세요.

In [ ]:
# 1) 받을 범위 선택 — 필요한 것만 True
GET_LABELS = True
GET_VAL    = True
GET_TRAIN  = False   # 약 24GB, Colab 디스크·시간 여유 있을 때

DATASET_KEY = 149
FILES = {
    # 구분: [(AI Hub 파일키, 저장 하위폴더, 파일명), ...]
    "labels": [
        (396903, "train/labels", "Apple_fuji_L.zip"), (396904, "train/labels", "Apple_fuji_M.zip"),
        (396905, "train/labels", "Apple_fuji_S.zip"), (396906, "train/labels", "Apple_yanggwang_L.zip"),
        (396907, "train/labels", "Apple_yanggwang_M.zip"), (396908, "train/labels", "Apple_yanggwang_S.zip"),
        (397011, "val/labels", "Apple_fuji_L.zip"), (397012, "val/labels", "Apple_fuji_M.zip"),
        (397013, "val/labels", "Apple_fuji_S.zip"), (397014, "val/labels", "Apple_yanggwang_L.zip"),
        (397015, "val/labels", "Apple_yanggwang_M.zip"), (397016, "val/labels", "Apple_yanggwang_S.zip"),
    ],
    "val": [
        (397065, "val/images", "Apple_fuji_L.zip"), (397066, "val/images", "Apple_fuji_M.zip"),
        (397067, "val/images", "Apple_fuji_S.zip"), (397068, "val/images", "Apple_yanggwang_L.zip"),
        (397069, "val/images", "Apple_yanggwang_M.zip"), (397070, "val/images", "Apple_yanggwang_S.zip"),
    ],
    "train": [
        (396957, "train/images", "Apple_fuji_L.zip"), (396958, "train/images", "Apple_fuji_M.zip"),
        (396959, "train/images", "Apple_fuji_S.zip"), (396960, "train/images", "Apple_yanggwang_L.zip"),
        (396961, "train/images", "Apple_yanggwang_M.zip"), (396962, "train/images", "Apple_yanggwang_S.zip"),
    ],
}
selected = (FILES["labels"] if GET_LABELS else []) + (FILES["val"] if GET_VAL else []) \
         + (FILES["train"] if GET_TRAIN else [])
print(f"받을 파일 {len(selected)}개")

In [ ]:
# 2) 공유 드라이브 연결
from google.colab import drive
drive.mount('/content/drive')

import os
DEST = '/content/drive/MyDrive/smart_apple_apc/data/raw/aihub_qc_apple'
assert os.path.isdir('/content/drive/MyDrive/smart_apple_apc'), \
    "내 드라이브에 smart_apple_apc 폴더가 없습니다. 드라이브 소유자 계정으로 실행하세요."
os.makedirs(DEST, exist_ok=True)
print("저장 위치:", DEST)

In [ ]:
# 3) aihubshell 설치
!curl -s -o /usr/local/bin/aihubshell https://api.aihub.or.kr/api/aihubshell.do
!chmod +x /usr/local/bin/aihubshell
!aihubshell -help | head -20

In [ ]:
# 4) API key 입력 (화면·노트북에 저장되지 않음)
from getpass import getpass
os.environ['AIHUB_APIKEY'] = getpass('AI Hub API key: ')

In [ ]:
# 5) 다운로드 → 분할 파일 병합 → 드라이브로 복사
import glob, shutil, subprocess

WORK = '/content/aihub_tmp'
done = []
for key, sub, name in selected:
    target = os.path.join(DEST, sub, name)
    if os.path.exists(target):
        print(f"⏭  이미 있음: {sub}/{name}")
        continue
    shutil.rmtree(WORK, ignore_errors=True); os.makedirs(WORK)
    print(f"⬇  {sub}/{name} (filekey {key})")
    r = subprocess.run(
        f"aihubshell -mode d -datasetkey {DATASET_KEY} -filekey {key} "
        f"-aihubapikey \"$AIHUB_APIKEY\"",
        shell=True, cwd=WORK, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:]); raise RuntimeError("다운로드 실패 — 승인 여부·API key 확인")
    # 분할 압축(.zip.part*) 병합
    parts = sorted(glob.glob(f"{WORK}/**/{name}.part*", recursive=True),
                   key=lambda p: int(p.rsplit('.part', 1)[1]))
    whole = glob.glob(f"{WORK}/**/{name}", recursive=True)
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if parts:
        with open(target + '.tmp', 'wb') as out:
            for p in parts:
                with open(p, 'rb') as f: shutil.copyfileobj(f, out, 16 * 1024 * 1024)
        os.replace(target + '.tmp', target)
    elif whole:
        shutil.copy(whole[0], target)
    else:
        raise RuntimeError(f"받은 파일을 찾지 못함: {name}")
    done.append(target); print(f"✅ 저장: {target}")
shutil.rmtree(WORK, ignore_errors=True)
print(f"\n완료 {len(done)}개")

In [ ]:
# 6) 확인: 압축 파일 무결성 + 이미지 수
import zipfile
for z in sorted(glob.glob(f"{DEST}/**/*.zip", recursive=True)):
    with zipfile.ZipFile(z) as f:
        bad = f.testzip()
        n = sum(1 for i in f.infolist() if not i.is_dir())
    print(f"{'✅' if bad is None else '❌'} {os.path.relpath(z, DEST):40s} 파일 {n:6d}개  {os.path.getsize(z)/1e9:6.2f} GB")

## 끝난 뒤
- `data/README.md` 의 **데이터셋 기록표**에 날짜·수량을 적어 PR로 올리기
- 팀원은 드라이브 `공유 문서함 → smart_apple_apc → data/raw/aihub_qc_apple` 에서 받거나, Colab에서 마운트해 바로 사용
- 압축은 학습할 때 Colab 로컬(`/content`)에 풀어서 쓰기 (드라이브에 풀면 매우 느림)